# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
# TODO
df['revenue'] = df['qty'] * df['price']
df.head()

print("Number of rows: " + str(len(df))) # num of rows for debug
print("Total Revenue: " + str(df['revenue'].sum())) # Total revenue for all products sold
print("Total Units: " + str(df['qty'].sum())) # Number of all products sold

Number of rows: 400
Total Revenue: 8520.0
Total Units: 783


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
# TODO
by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False).reset_index()
by_category['share_of_total'] = (by_category['revenue'] / df['revenue'].sum()) * 100
display(by_category)

# The highest revenue category is Food, which made up 50.39% of the total with 4293 in revenue.

,category,revenue,share_of_total
0,Food,4293.0,50.387324
1,Merch,1771.5,20.792254
2,Drink,1554.0,18.239437
3,RainGear,901.5,10.580986


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
# TODO
by_vendor = df.groupby('vendor_id')['revenue'].agg(['count', 'mean']).sort_values('mean', ascending=False).reset_index()
display(by_vendor)

# Two rows showing the order count for 4 vendors and the averages.

,vendor_id,count,mean
0,V-01,94,22.595745
1,V-18,108,21.750000
2,V-05,93,20.580645
3,V-10,105,20.314286


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
# TODO
merch_revenue = df[df['category'] == 'Merch']['revenue'].sum()
total_revenue = df['revenue'].sum()
merch_share = (merch_revenue / total_revenue) * 100

print("Merch Share of Revenue: " + str(round(merch_share, 1)) + "%")
# Printed the percentage of total revenue that came from Merch sales.

Merch Share of Revenue: 20.8%


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
joined = pd.merge(df, vendor_names, on='vendor_id', how='left', validate='many_to_one')

unmatched_vendor_id = joined[joined['vendor_name'].isna()]['vendor_id'].unique()
print(f"Unmatched Vendor ID: {unmatched_vendor_id[0] if len(unmatched_vendor_id) > 0 else 'None'}")

original_row_count = len(df)
merged_row_count = len(joined)
original_total_revenue = df['revenue'].sum()
merged_total_revenue = joined['revenue'].sum()

print(f"Original row count: {original_row_count}, Merged row count: {merged_row_count}")
print(f"Original total revenue: {original_total_revenue:.2f}, Merged total revenue: {merged_total_revenue:.2f}")
display(joined.head())

Unmatched Vendor ID: V-18
Original row count: 400, Merged row count: 400
Original total revenue: 8520.00, Merged total revenue: 8520.00


,vendor_id,category,qty,price,revenue,vendor_name
0,V-10,Drink,2,24.0,48.0,Cav Merch North
1,V-18,RainGear,1,12.0,12.0,NaN
2,V-18,Drink,3,4.5,13.5,NaN
3,V-10,Food,2,12.0,24.0,Cav Merch North
4,V-18,Drink,3,7.5,22.5,NaN


**The unmatched vendor, and what I did about it:** The vendor with ID V-18 was found in the orders df but not in the vendor_names lookup table, and when doing the left merge, entries for V-18 resulted in NaN for vendor_name. This is okay for a left join, b/c it keeps all original order data while indicating missing vendor name information. The row count was still at 400 and the total revenue at 8520.00, which shows the merge did not alter the dataset in terms of these key metrics.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
# TODO
joined['vendor_name'] = joined['vendor_name'].fillna('Unknown (V-18)')
pivot_table = pd.pivot_table(
    joined,
    values='revenue',
    index='vendor_name',
    columns='category',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
)
display(pivot_table)

# The pivot table shows all vendors, their revenue by category, and total revenue.

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown (V-18),582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a. Vendors should prioritize food and merch sales. Food sales generated around 50.4% of the total revenue, showing that it is a high-demand category for all sales. Similarly, merch contributed around 20.8% of total revenue for vendors.

b. The pivot table showing the revenue by vendor and category is the least trustworthy, because the pivot causes 108 orders to be left out entirely, which is about 27% of revenue.